In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
CATALOGO, SCHEMA = "fiap_analytics", "bronze"
BRONZE_TABLE = f"{CATALOGO}.{SCHEMA}.cemaden_rainfall_readings"
ENDPOINT = "/pcds/pcds-dados-pg"
BATCH_ID = str(uuid.uuid4())

SCHEMA_ORIGEM = T.StructType([
    T.StructField(c, T.StringType())
    for c in ["cidade", "codestacao", "datahora", "id_sensor", "latitude", "longitude",
              "nome", "offset", "qualificacao", "uf", "valor"]
])

# Bloco padrão de log, igual em todos os notebooks de carga. Cada origem tem a sua tabela
# (cemaden_ingestion_log, sibh_ingestion_log, geosampa_ingestion_log), todas com este formato.
LOG_TABLE = "fiap_analytics.bronze.cemaden_ingestion_log"
FONTE = "cemaden_rainfall_readings"
ETAPA = "bronze"

LOG_SCHEMA = T.StructType([
    T.StructField("batch_id", T.StringType()),
    T.StructField("fonte", T.StringType()),
    T.StructField("etapa", T.StringType()),
    T.StructField("janela", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("registros", T.LongType()),
    T.StructField("paginas", T.IntegerType()),
    T.StructField("duplicados", T.LongType()),
    T.StructField("arquivo", T.StringType()),
    T.StructField("sha256", T.StringType()),
    T.StructField("duracao_s", T.DoubleType()),
    T.StructField("iniciado_em", T.TimestampType()),
    T.StructField("finalizado_em", T.TimestampType()),
    T.StructField("mensagem", T.StringType()),
])

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
  batch_id      STRING    COMMENT 'Identificador da execução',
  fonte         STRING    COMMENT 'Conjunto de dados',
  etapa         STRING    COMMENT 'landing ou bronze',
  janela        STRING    COMMENT 'Período processado: mês, data da coleta ou ano do arquivo',
  status        STRING    COMMENT 'SUCESSO, PARCIAL, SEM_DADOS ou FALHA',
  registros     BIGINT,
  paginas       INT       COMMENT 'Chamadas à API',
  duplicados    BIGINT    COMMENT 'Duplicidades pela chave natural do conjunto',
  arquivo       STRING,
  sha256        STRING,
  duracao_s     DOUBLE,
  iniciado_em   TIMESTAMP,
  finalizado_em TIMESTAMP,
  mensagem      STRING
) COMMENT 'Controle e checkpoint das cargas do projeto ChuvaSegura, uma tabela por origem'
""")

# Para antes de gravar se a tabela existente não estiver no formato padrão.
_formato_atual = [(f.name, f.dataType) for f in spark.table(LOG_TABLE).schema]
if _formato_atual != [(f.name, f.dataType) for f in LOG_SCHEMA]:
    raise RuntimeError(f"{LOG_TABLE} está fora do formato padrão: {_formato_atual}")


def registrar(janela, status, iniciado_em, registros=None, paginas=None, duplicados=None,
              arquivo=None, sha256=None, mensagem=None) -> None:
    """Grava uma linha no log da origem. Status: SUCESSO, PARCIAL, SEM_DADOS ou FALHA."""
    finalizado_em = datetime.now(timezone.utc)
    linha = {
        "batch_id": BATCH_ID, "fonte": FONTE, "etapa": ETAPA, "janela": janela, "status": status,
        "registros": registros, "paginas": paginas, "duplicados": duplicados, "arquivo": arquivo,
        "sha256": sha256, "duracao_s": (finalizado_em - iniciado_em).total_seconds(),
        "iniciado_em": iniciado_em, "finalizado_em": finalizado_em,
        "mensagem": str(mensagem)[:1000] if mensagem else None,
    }
    spark.createDataFrame([linha], LOG_SCHEMA).write.mode("append").saveAsTable(LOG_TABLE)

pendentes = spark.sql(f"""
WITH landing AS (
  SELECT janela,
         max_by(arquivo, finalizado_em)   AS arquivo,
         max_by(registros, finalizado_em) AS registros,
         max(finalizado_em)               AS landing_em
  FROM {LOG_TABLE}
  WHERE fonte = '{FONTE}' AND etapa = 'landing' AND status IN ('SUCESSO', 'PARCIAL') AND registros > 0
  GROUP BY janela
),
bronze AS (
  SELECT janela, max(finalizado_em) AS bronze_em
  FROM {LOG_TABLE}
  WHERE fonte = '{FONTE}' AND etapa = 'bronze' AND status = 'SUCESSO'
  GROUP BY janela
)
SELECT l.janela, l.arquivo, l.registros
FROM landing l LEFT JOIN bronze b USING (janela)
WHERE b.bronze_em IS NULL OR b.bronze_em < l.landing_em
ORDER BY l.janela
""").collect()

print(f"{len(pendentes)} meses pendentes:", [p.janela for p in pendentes])

falhas = []
for p in pendentes:
    iniciado_em = datetime.now(timezone.utc)
    try:
        df = (
            spark.read.schema(SCHEMA_ORIGEM)
            .option("rescuedDataColumn", "_rescued_data")  # guarda campos novos que a API passe a enviar
            .json(p.arquivo)
            .select(
                "*",
                F.col("_metadata.file_path").alias("_source_file"),
                F.lit(ENDPOINT).alias("_source_endpoint"),
                F.lit(p.janela).alias("_window"),
                F.lit(int(p.janela[:4])).alias("_window_year"),
                F.lit(int(p.janela[4:])).alias("_window_month"),
                F.lit(BATCH_ID).alias("_batch_id"),
                F.current_timestamp().alias("_ingested_at"),
            )
        )
        lidos = df.count()
        if lidos != p.registros:
            raise RuntimeError(f"arquivo tem {lidos} linhas, a landing registrou {p.registros}")

        if not spark.catalog.tableExists(BRONZE_TABLE):
            df.write.format("delta").partitionBy("_window_year").saveAsTable(BRONZE_TABLE)
            spark.sql(f"COMMENT ON TABLE {BRONZE_TABLE} IS "
                      "'Leituras brutas da API PED do CEMADEN (pcds-dados-pg), município de São Paulo, horários em UTC. "
                      "Fonte: rede observacional do CEMADEN/MCTIC.'")
        else:
            (df.write.format("delta").mode("overwrite")
             .option("replaceWhere", f"_window = '{p.janela}'")
             .saveAsTable(BRONZE_TABLE))

        registrar(p.janela, "SUCESSO", iniciado_em, registros=lidos, arquivo=p.arquivo)
        print(f"{p.janela}: {lidos} linhas carregadas")
    except Exception as e:
        registrar(p.janela, "FALHA", iniciado_em, arquivo=p.arquivo, mensagem=str(e))
        falhas.append(p.janela)
        print(f"{p.janela}: FALHA {e}")

if falhas:
    raise RuntimeError(f"Meses com falha na Bronze: {falhas}")

display(spark.sql(f"""
SELECT _window, count(*) AS linhas, count(DISTINCT codestacao) AS estacoes,
       min(datahora) AS primeira_leitura, max(datahora) AS ultima_leitura,
       count_if(_rescued_data IS NOT NULL) AS linhas_com_campos_novos
FROM {BRONZE_TABLE}
GROUP BY _window
ORDER BY _window DESC
"""))